In [1]:
cd /home/nampv1/projects/vnpost_asr/

/home/nampv1/projects/vnpost_asr


In [2]:
hf_raw_data_dir = '/media/nampv1/hdd/data/ASR-VLSP2020-VINAI-100H/raw/hf'

In [3]:
from datasets import load_from_disk, load_dataset

/home/nampv1/anaconda3/envs/asr/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
dataset = load_from_disk(hf_raw_data_dir)

In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 56427
    })
})

### Get Druations

In [ ]:
from datasets import load_from_disk
import numpy as np

# load dataset
ds = dataset

# compute durations
def get_duration(example):
    sr = example["audio"]["sampling_rate"]
    n_samples = len(example["audio"]["array"])
    example["duration"] = n_samples / sr
    return example

ds_with_dur = ds.map(get_duration)

# total duration
durations = ds_with_dur["train"]["duration"]
total_seconds = np.sum(durations)
total_hours = total_seconds / 3600

print(f"Total duration: {total_seconds:.2f} seconds ({total_hours:.2f} hours)")


Map:  14%|█▍        | 7982/56427 [08:43<04:36, 175.47 examples/s]  

In [ ]:
#

### Split into train, dev, test splits

In [8]:
split_ds = dataset["train"].train_test_split(test_size=5000, seed=202508)


In [10]:

# first split: train + test
split = dataset["train"].train_test_split(test_size=5000, seed=42)

# second split: split train into train + dev
train_dev = split["train"].train_test_split(test_size=5000, seed=42)

# build final datasetdict
final_ds = {
    "train": train_dev["train"],
    "dev": train_dev["test"],
    "test": split["test"]
}

from datasets import DatasetDict
final_ds = DatasetDict(final_ds)

print(final_ds)

DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 46427
    })
    dev: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 5000
    })
})


In [11]:
final_ds.save_to_disk("/media/nampv1/hdd/data/ASR-VLSP2020-VINAI-100H/raw/hf1")

Saving the dataset (3/3 shards): 100%|██████████| 5000/5000 [00:19<00:00, 251.44 examples/s] 


In [15]:
from datasets import load_from_disk, DatasetDict

def make_splits(dataset, test_size=0.1, dev_size=0.1, seed=42):
    """
    Generate train/dev/test splits from a Hugging Face dataset.

    Args:
        dataset (DatasetDict): Hugging Face dataset with a 'train' split.
        test_size (int|float): Number of samples or ratio for test split.
        dev_size (int|float): Number of samples or ratio for dev split (from train).
        seed (int): Random seed for reproducibility.

    Returns:
        DatasetDict with train, dev, test splits.
    """
    # split off test
    split = dataset["train"].train_test_split(test_size=test_size, seed=seed)

    # split train into train + dev
    train_dev = split["train"].train_test_split(test_size=dev_size, seed=seed)

    return DatasetDict({
        "train": train_dev["train"],
        "dev": train_dev["test"],
        "test": split["test"]
    })


# example usage
from datasets import load_from_disk
ds = load_from_disk("/media/nampv1/hdd/data/ASR-VLSP2020-VINAI-100H/raw/hf")
final_ds = make_splits(ds, test_size=0.1, dev_size=0.1)
print(final_ds)


DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 45705
    })
    dev: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 5079
    })
    test: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 5643
    })
})
